In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

In [2]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 本地数据集：repo_id 为子目录名，root 为父目录
dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
    )
dataset

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [3]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot_policy_demo import DemoConfig

register_third_party_plugins()  # 必须：注册 lerobot_policy_my_policy 插件

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 方式 A：从零初始化（还没有训练好的 checkpoint）
config = DemoConfig(device=str(device))
policy = make_policy(config, ds_meta=dataset.meta).eval()
policy

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


DemoPolicy(
  (model): Sequential(
    (0): Linear(in_features=9, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=12, bias=True)
  )
)

In [4]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

print("policy type:", policy.config.type)
print("input features:", list(policy.config.input_features.keys()))

policy type: demo_policy
input features: ['observation.state', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image']


### 推理验证

In [12]:
batch = preprocess(dataset[0])
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }
batch = to_device(batch)

In [6]:
with torch.inference_mode():
    pred_action_raw = policy.select_action(batch)
pred_action_raw

tensor([[-0.4334,  0.0195, -0.7798,  0.2288,  0.1536, -0.2549, -0.3083, -0.0961,
          0.2312,  0.1069,  0.1702,  0.2884]], device='cuda:0')

### 前向传播

In [7]:
with torch.inference_mode(): # 不进行反向传播
    loss, output_dict = policy.forward(batch)
print("loss:", loss.item() if hasattr(loss, "item") else loss)
print("output keys:", output_dict.keys() if isinstance(output_dict, dict) else type(output_dict))

loss: 0.9559996724128723
output keys: dict_keys(['mse'])


In [8]:
policy.config

DemoConfig(n_obs_steps=1, input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(9,)), 'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(12,))}, device='cuda', use_amp=False, use_peft=False, push_to_hub=True, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, chunk_size=50, n_action_steps=50, hidden_dim=256, normalization_mapping={'VISUAL': <NormalizationMode.MEAN_STD: 'MEAN_STD'>, 'STATE': <NormalizationMode.MIN_MAX: 'MIN_MAX'>, 'ACTION': <NormalizationMode.MIN_MAX: 'MIN_MAX'>})

DemoConfig(n_obs_steps=1, 
input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(9,)), 
'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)),
 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 
 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224))}, 
 output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(12,))}, device='cuda', use_amp=False, use_peft=False, push_to_hub=True, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, chunk_size=50, n_action_steps=50, hidden_dim=256, normalization_mapping={'VISUAL': <NormalizationMode.MEAN_STD: 'MEAN_STD'>, 'STATE': <NormalizationMode.MIN_MAX: 'MIN_MAX'>, 'ACTION': <NormalizationMode.MIN_MAX: 'MIN_MAX'>})

In [10]:
policy.config.push_to_hub

True